In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/to-read/to_read_clean.csv
/kaggle/input/book-recommender/books_clean.csv
/kaggle/input/book-tag/book_tag_tfidf.npz
/kaggle/input/rating-clean/ratings_clean.csv


In [6]:
import sys
print(sys.version)

3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]


In [7]:
!pip install lightfm

# **Import Libraries**

In [8]:
import pandas as pd
import numpy as np
from scipy import sparse
from lightfm import LightFM
from lightfm.evaluation import precision_at_k, recall_at_k
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from fuzzywuzzy import process
from sklearn.metrics.pairwise import cosine_similarity


/usr/local/lib/python3.11/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


# **Load Data**

In [9]:
books = pd.read_csv('/kaggle/input/book-recommender/books_clean.csv')
ratings = pd.read_csv('/kaggle/input/rating-clean/ratings_clean.csv')
to_read = pd.read_csv('/kaggle/input/to-read/to_read_clean.csv')
book_tags_tfidf = sparse.load_npz('/kaggle/input/book-tag/book_tag_tfidf.npz')

In [10]:
books.head()

,book_id,goodreads_book_id,title,authors,average_rating,ratings_count_all_editions,text_reviews_count,image_url,small_image_url,popularity,avg_rating,p_rating_1,p_rating_2,p_rating_3,p_rating_4,p_rating_5
0,1,2767052,The Hunger Games,Suzanne Collins,4.34,4942365,155254,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...,4942365,4.34,0.013499,0.025886,0.113325,0.299716,0.547575
1,2,3,Harry Potter and the Philosopher's Stone,"J.K. Rowling, Mary GrandPré",4.44,4800065,75867,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...,4800065,4.44,0.015730,0.021182,0.094795,0.240896,0.627396
2,3,41865,Twilight,Stephenie Meyer,3.57,3916824,95009,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...,3916824,3.57,0.116470,0.111519,0.202541,0.223414,0.346056
3,4,2657,To Kill a Mockingbird,Harper Lee,4.25,3340896,72586,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...,3340896,4.25,0.018087,0.035145,0.133747,0.299905,0.513116
4,5,4671,The Great Gatsby,F. Scott Fitzgerald,3.89,2773745,51992,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...,2773745,3.89,0.031090,0.071247,0.218534,0.337454,0.341675


In [11]:
ratings.head()

,user_id,book_id,rating
0,1,258,5
1,2,4081,4
2,2,260,5
3,2,9296,5
4,2,2318,3


In [12]:
to_read.head()

,user_id,book_id
0,9,8
1,15,398
2,15,275
3,37,7173
4,34,380


In [13]:
book_tags_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 854820 stored elements and shape (10000, 14235)>

# **Prepare IDS**

In [14]:
user_encoder = LabelEncoder()
ratings['user_idx'] = user_encoder.fit_transform(ratings['user_id'])
to_read['user_idx'] = user_encoder.transform(to_read['user_id'])

book_encoder = LabelEncoder()
ratings['book_idx'] = book_encoder.fit_transform(ratings['book_id'])
to_read['book_idx'] = book_encoder.transform(to_read['book_id'])
books['book_idx'] = book_encoder.transform(books['book_id'])


In [15]:
num_users = ratings['user_idx'].nunique()
num_items = ratings['book_idx'].nunique()

In [16]:
num_users

53424

In [17]:
num_items 

10000

# **Train/Test**

In [18]:
train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)

In [19]:
def build_interaction_matrix(df):
    matrix = sparse.lil_matrix((num_users, num_items))
    for row in df.itertuples():
        matrix[row.user_idx, row.book_idx] = row.rating
    return matrix.tocsr()

train_matrix = build_interaction_matrix(train_data)
test_matrix = build_interaction_matrix(test_data)

In [20]:
# for row in to_read.itertuples():
#     if train_matrix[row.user_idx, row.book_idx] == 0:
#         train_matrix[row.user_idx, row.book_idx] = 1

In [21]:

from scipy.sparse import csr_matrix


all_data = pd.concat([
    ratings[['user_idx','book_idx']],
    to_read[['user_idx','book_idx']]
])


train_matrix = csr_matrix(
    (np.ones(len(all_data)), (all_data['user_idx'], all_data['book_idx'])),
    shape=(num_users, num_items)
)

# **LightFM Hybrid Model**

In [22]:
model = LightFM(loss='bpr')
model.fit(interactions=train_matrix, item_features=book_tags_tfidf, epochs=5, num_threads=4)

In [ ]:
precision = precision_at_k(model, test_matrix, item_features=book_tags_tfidf, k=10).mean()
recall = recall_at_k(model, test_matrix, item_features=book_tags_tfidf, k=10).mean()
print(f"Precision@10: {precision:.4f}")
print(f"Recall@10: {recall:.4f}")

**Recommendation function**

In [ ]:
def recommend_books_by_name(book_name, top_n=10):
    best_match = process.extractOne(book_name, books['title'])[0]
    book_idx = books[books['title'] == best_match]['book_idx'].values[0]
    book_vec = book_tags_tfidf[book_idx]
    sims = cosine_similarity(book_vec, book_tags_tfidf).flatten()
    top_indices = sims.argsort()[::-1]
    top_indices = top_indices[top_indices != book_idx][:top_n]
    recommended = books.iloc[top_indices][['title', 'authors']]
    return recommended

In [ ]:
def recommend_books_for_user(user_id, top_n=10):
    try:
        user_idx = user_encoder.transform([user_id])[0]
    except:
        return pd.DataFrame(columns=['title','authors'])
    scores = model.predict(user_idx, np.arange(num_items), item_features=book_tags_tfidf)
    top_indices = np.argsort(-scores)[:top_n]
    recommended = books.iloc[top_indices][['title', 'authors']]
    return recommended

**Try Recommendations**

In [ ]:
print("Recommendations by Book")
print(recommend_books_by_name("Harry Potter", top_n=5))

print("\nRecommendations for User")
print(recommend_books_for_user(user_id=15, top_n=5))

In [ ]:
import pickle


with open('lightfm_hybrid_model.pkl', 'wb') as f:
    pickle.dump(model, f)


with open('user_encoder.pkl', 'wb') as f:
    pickle.dump(user_encoder, f)

with open('book_encoder.pkl', 'wb') as f:
    pickle.dump(book_encoder, f)